In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install    mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 65.8 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [ ]:
import os, glob, random, gc
import numpy as np
import pandas as pd
import mne
import scipy.signal
import scipy.stats
from tqdm.auto import tqdm
import warnings
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')


TARGET_CLASSES = ['fnsz', 'gnsz', 'cpsz', 'bckg']

CHANNELS_ORDER = [
    'FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4',
    'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6',
    'A1', 'A2', 'FZ', 'CZ'
]

WINDOW_SIZE_SEC   = 10.0
TARGET_SFREQ      = 128.0
POINTS_PER_WINDOW = int(WINDOW_SIZE_SEC * TARGET_SFREQ)
OVERLAP_POINTS    = int(POINTS_PER_WINDOW * 0.5)

MAX_BKG_WINDOWS = 100

SAVE_DIR = '/content/drive/MyDrive/gp_dataset/fusion_raw_and_5x5_v4'
os.makedirs(SAVE_DIR, exist_ok=True)


def extract_advanced_features_vectorized(batch_data, sfreq=128.0):
    B, C, T_pts = batch_data.shape
    num_features = 15

    mean_val = np.mean(batch_data, axis=-1)
    std_val  = np.std(batch_data, axis=-1)
    skew_val = scipy.stats.skew(batch_data, axis=-1)
    kurt_val = scipy.stats.kurtosis(batch_data, axis=-1)

    freqs, psd = scipy.signal.welch(batch_data, sfreq, nperseg=256, axis=-1)
    bands = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 12), 'beta': (12, 30)}
    powers = []
    for low, high in bands.values():
        idx_band = np.logical_and(freqs >= low, freqs <= high)
        powers.append(np.trapz(psd[:, :, idx_band], freqs[idx_band], axis=-1))

    powers = np.array(powers)
    total_power = np.sum(powers, axis=0) + 1e-8
    rel_powers = powers / total_power
    delta_theta_ratio = powers[0] / (powers[1] + 1e-8)

    d1 = np.diff(batch_data, axis=-1)
    d2 = np.diff(d1, axis=-1)
    var_zero = np.var(batch_data, axis=-1)
    var_d1   = np.var(d1, axis=-1)
    var_d2   = np.var(d2, axis=-1)

    act = var_zero
    mob = np.sqrt(var_d1 / np.where(var_zero == 0, 1e-8, var_zero))
    comp = np.sqrt(var_d2 / np.where(var_d1 == 0, 1e-8, var_d1)) / np.where(mob == 0, 1e-8, mob)

    psd_norm = psd / (np.sum(psd, axis=-1, keepdims=True) + 1e-8)
    entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-8), axis=-1)

    mean_corr = np.zeros((B, C))
    analytic_signal = scipy.signal.hilbert(batch_data, axis=-1)
    Z = analytic_signal / (np.abs(analytic_signal) + 1e-8)

    for b in range(B):
        corr_matrix = np.corrcoef(batch_data[b])
        mean_corr[b] = np.mean(np.abs(corr_matrix), axis=1)

    plv_matrix = np.abs(Z @ np.swapaxes(Z.conj(), 1, 2)) / T_pts
    mean_plv = (np.sum(plv_matrix, axis=-1) - 1.0) / (C - 1)

    features_per_channel = np.stack((
        mean_val, std_val, skew_val, kurt_val,
        rel_powers[0], rel_powers[1], rel_powers[2], rel_powers[3],
        delta_theta_ratio, act, mob, comp, entropy, mean_corr, mean_plv
    ), axis=-1)

    spatial_map = np.zeros((B, 5, 5, num_features))
    mapping = {
        (0,1): 0,  (0,2): 18, (0,3): 1,
        (1,0): 10, (1,1): 2,  (1,3): 3,  (1,4): 11,
        (2,0): 12, (2,1): 4,  (2,2): 19, (2,3): 5,  (2,4): 13,
        (3,0): 14, (3,1): 6,  (3,3): 7,  (3,4): 15,
        (4,0): 16, (4,1): 8,  (4,3): 9,  (4,4): 17
    }
    for (row, col), ch_idx in mapping.items():
        spatial_map[:, row, col, :] = features_per_channel[:, ch_idx, :]

    return spatial_map


TRAIN_FOLDER = '/content/drive/MyDrive/gp_dataset/seizure_v2.0.3/edf/train'
csv_files    = glob.glob(f'{TRAIN_FOLDER}/**/*.csv', recursive=True)

print("Pre-scanning patients...")
patient_files    = {}
patient_has_cpsz = {}

for csv in csv_files:
    pat_id = os.path.basename(csv).split('_')[0]
    if pat_id not in patient_files:
        patient_files[pat_id]    = []
        patient_has_cpsz[pat_id] = False
    patient_files[pat_id].append(csv)

    if not patient_has_cpsz[pat_id]:
        try:
            df_ann = pd.read_csv(csv, skiprows=6, header=None)
            for _, r in df_ann.iterrows():
                if str(r[3]).lower().strip() == 'cpsz':
                    patient_has_cpsz[pat_id] = True
                    break
        except:
            pass

cpsz_patients  = [p for p, has in patient_has_cpsz.items() if has]
other_patients = [p for p, has in patient_has_cpsz.items() if not has]

random.seed(42)
random.shuffle(cpsz_patients)
random.shuffle(other_patients)

c_val_split  = max(1, int(0.15 * len(cpsz_patients)))
c_test_split = max(1, int(0.15 * len(cpsz_patients)))

val_cpsz   = cpsz_patients[:c_val_split]
test_cpsz  = cpsz_patients[c_val_split: c_val_split + c_test_split]
train_cpsz = cpsz_patients[c_val_split + c_test_split:]

o_val_split  = int(0.10 * len(other_patients))
o_test_split = int(0.10 * len(other_patients))

val_other   = other_patients[:o_val_split]
test_other  = other_patients[o_val_split: o_val_split + o_test_split]
train_other = other_patients[o_val_split + o_test_split:]

train_patients = set(train_cpsz + train_other)
val_patients   = set(val_cpsz   + val_other)
test_patients  = set(test_cpsz  + test_other)


def process_single_csv(csv_path):
    raw_windows_list, labels_list = [], []
    try:
        edf_path = csv_path.replace('.csv', '.edf').replace('.csv.tse', '.edf')
        if not os.path.exists(edf_path): return [], [], []

        df_ann = pd.read_csv(csv_path, skiprows=6, header=None)
        seizure_events = [(float(r[1]), float(r[2]), str(r[3]).lower().strip())
                          for _, r in df_ann.iterrows() if str(r[3]).lower().strip() in TARGET_CLASSES]
        if not seizure_events: return [], [], []

        raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
        actual_picks = [raw.ch_names.index(c) for target in CHANNELS_ORDER for c in raw.ch_names if f"EEG {target}-" in c]
        if len(actual_picks) != len(CHANNELS_ORDER): return [], [], []

        original_sfreq = raw.info['sfreq']
        file_bkg_count = 0

        for (s_start, s_end, s_label) in seizure_events:
            if s_label == 'bckg' and file_bkg_count >= MAX_BKG_WINDOWS: continue

            start_idx = int(s_start * original_sfreq)
            end_idx   = int(s_end   * original_sfreq)

            if s_label == 'bckg':
                rem = MAX_BKG_WINDOWS - file_bkg_count
                max_pts = int((WINDOW_SIZE_SEC + (rem - 1) * 5.0) * original_sfreq)
                if (end_idx - start_idx) > max_pts: end_idx = start_idx + max_pts

            if start_idx >= end_idx: continue

            data, _ = raw[actual_picks, start_idx:end_idx]
            if data.shape[1] == 0: continue

            if original_sfreq > TARGET_SFREQ:
                data = mne.filter.resample(data, up=TARGET_SFREQ, down=original_sfreq)
                sfreq = TARGET_SFREQ
            else:
                sfreq = original_sfreq

            data = mne.filter.notch_filter(data, sfreq, freqs=60.0, verbose=False)

            data = mne.filter.filter_data(data, sfreq, l_freq=1.0, h_freq=30.0, verbose=False)

            median_val = np.median(data, axis=1, keepdims=True)
            q75, q25 = np.percentile(data, [75 ,25], axis=1, keepdims=True)
            iqr_val = q75 - q25
            data = (data - median_val) / (iqr_val + 1e-8)

            data = np.clip(data, -5.0, 5.0)


            for win_start in range(0, data.shape[1] - POINTS_PER_WINDOW + 1, OVERLAP_POINTS):
                if s_label == 'bckg':
                    if file_bkg_count >= MAX_BKG_WINDOWS: break
                    file_bkg_count += 1

                window_data = data[:, win_start:win_start + POINTS_PER_WINDOW]
                raw_windows_list.append(window_data)
                labels_list.append(s_label)

    except Exception as e:
        pass

    if not raw_windows_list:
        return [], [], []

    raw_windows_arr = np.stack(raw_windows_list)
    X_eng_local = extract_advanced_features_vectorized(raw_windows_arr, sfreq=TARGET_SFREQ)

    X_raw_local = np.swapaxes(raw_windows_arr, 1, 2)

    return X_raw_local, X_eng_local, labels_list



import gc

def extract_for_split_chunked(patient_set, split_name, chunk_size=50):
    print(f"\nExtracting {split_name} split (Sequential Mode for Stability)...")
    split_csvs = [csv for csv in csv_files if os.path.basename(csv).split('_')[0] in patient_set]

    total_chunks = (len(split_csvs) // chunk_size) + 1

    for chunk_idx, i in enumerate(range(0, len(split_csvs), chunk_size)):
        if os.path.exists(f'{SAVE_DIR}/y_{split_name.lower()}_part{chunk_idx}.npy'):
            print(f"   Skipping chunk {chunk_idx + 1}/{total_chunks} (Already exists)...")
            continue

        chunk_csvs = split_csvs[i:i+chunk_size]
        print(f"\n   Processing chunk {chunk_idx + 1}/{total_chunks} ({len(chunk_csvs)} files)...")

        X_raw_list, X_eng_list, y_list = [], [], []

        for csv in tqdm(chunk_csvs, desc=f"Part {chunk_idx+1}", leave=False):
            try:
                raw_res, eng_res, labels_res = process_single_csv(csv)
                if len(labels_res) > 0:
                    X_raw_list.append(raw_res)
                    X_eng_list.append(eng_res)
                    y_list.extend(labels_res)
            except Exception as e:
                print(f"   Failed to process {os.path.basename(csv)}: {e}")
                continue

            gc.collect()

        if len(X_raw_list) > 0:
            X_raw_chunk = np.concatenate(X_raw_list, axis=0).astype(np.float32)
            X_eng_chunk = np.concatenate(X_eng_list, axis=0).astype(np.float32)
            y_chunk = np.array(y_list)

            np.save(f'{SAVE_DIR}/X_raw_{split_name.lower()}_part{chunk_idx}.npy', X_raw_chunk)
            np.save(f'{SAVE_DIR}/X_eng_{split_name.lower()}_part{chunk_idx}.npy', X_eng_chunk)
            np.save(f'{SAVE_DIR}/y_{split_name.lower()}_part{chunk_idx}.npy', y_chunk)

            print(f"   Saved Part {chunk_idx} -> Raw: {X_raw_chunk.shape} | Eng: {X_eng_chunk.shape}")

            del X_raw_chunk, X_eng_chunk, y_chunk, X_raw_list, X_eng_list, y_list
            gc.collect()

    print(f"{split_name} Split fully extracted!")

# extract_for_split_chunked(train_patients, "TRAIN", chunk_size=50)
extract_for_split_chunked(val_patients,   "VAL", chunk_size=50)
# extract_for_split_chunked(test_patients,  "TEST", chunk_size=10)

def extract_with_skip(patient_set, split_name, chunk_size=20, skip_chunks=[]):
    print(f"\nProcessing {split_name} (Chunk size: {chunk_size})...")
    split_csvs = [csv for csv in csv_files if os.path.basename(csv).split('_')[0] in patient_set]

    for chunk_idx, i in enumerate(range(0, len(split_csvs), chunk_size)):
        if chunk_idx in skip_chunks:
            print(f"   Manually SKIPPING chunk {chunk_idx} to avoid RAM crash.")
            continue

        target_file = f'{SAVE_DIR}/y_{split_name.lower()}_part{chunk_idx}.npy'
        if os.path.exists(target_file):
            print(f"   Skipping chunk {chunk_idx} (Already exists).")
            continue

        chunk_csvs = split_csvs[i:i+chunk_size]
        print(f"   Processing chunk {chunk_idx}...")

        X_raw_list, X_eng_list, y_list = [], [], []
        for csv in tqdm(chunk_csvs, desc=f"Part {chunk_idx}", leave=False):
            raw_res, eng_res, labels_res = process_single_csv(csv)
            if len(labels_res) > 0:
                X_raw_list.append(raw_res)
                X_eng_list.append(eng_res)
                y_list.extend(labels_res)

        if X_raw_list:
            np.save(f'{SAVE_DIR}/X_raw_{split_name.lower()}_part{chunk_idx}.npy', np.concatenate(X_raw_list, axis=0))
            np.save(f'{SAVE_DIR}/X_eng_{split_name.lower()}_part{chunk_idx}.npy', np.concatenate(X_eng_list, axis=0))
            np.save(f'{SAVE_DIR}/y_{split_name.lower()}_part{chunk_idx}.npy', np.array(y_list))

        gc.collect()

extract_with_skip(test_patients, "TEST", chunk_size=20, skip_chunks=[5])

Pre-scanning patients...

Extracting VAL split (Sequential Mode for Stability)...
   Skipping chunk 1/9 (Already exists)...
   Skipping chunk 2/9 (Already exists)...
   Skipping chunk 3/9 (Already exists)...
   Skipping chunk 4/9 (Already exists)...
   Skipping chunk 5/9 (Already exists)...
   Skipping chunk 6/9 (Already exists)...
   Skipping chunk 7/9 (Already exists)...
   Skipping chunk 8/9 (Already exists)...
   Skipping chunk 9/9 (Already exists)...
VAL Split fully extracted!

Processing TEST (Chunk size: 20)...
   Skipping chunk 0 (Already exists).
   Skipping chunk 1 (Already exists).
   Skipping chunk 2 (Already exists).
   Skipping chunk 3 (Already exists).
   Skipping chunk 4 (Already exists).
   Manually SKIPPING chunk 5 to avoid RAM crash.
   Skipping chunk 6 (Already exists).
   Skipping chunk 7 (Already exists).
   Skipping chunk 8 (Already exists).
   Skipping chunk 9 (Already exists).
   Skipping chunk 10 (Already exists).
   Skipping chunk 11 (Already exists).
   Skip